In [12]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [32]:
df = pd.read_csv("../datas/all_sites_combined.csv")

In [33]:
df.head()

,timestamp,site_id,site_type,site_name,consumption_kwh,consumption_euros,temperature_celsius,humidity_percent,solar_irradiance_wm2,hour,day_of_week,day_name,month,is_weekend,is_working_hours
0,2023-01-01 00:00:00,SITE001,office,Bureau Tertiaire,85.82,12.87,10.6,76.7,0.0,0,6,Sunday,1,1,0
1,2023-01-01 01:00:00,SITE001,office,Bureau Tertiaire,89.84,13.48,12.0,70.6,0.0,1,6,Sunday,1,1,0
2,2023-01-01 02:00:00,SITE001,office,Bureau Tertiaire,80.62,12.09,13.8,62.2,0.0,2,6,Sunday,1,1,0
3,2023-01-01 03:00:00,SITE001,office,Bureau Tertiaire,92.67,13.90,9.4,66.0,0.0,3,6,Sunday,1,1,0
4,2023-01-01 04:00:00,SITE001,office,Bureau Tertiaire,72.50,10.88,14.8,67.2,41.5,4,6,Sunday,1,1,0


In [34]:
df.shape

(122647, 15)

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 122647 entries, 0 to 122646
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   timestamp             122647 non-null  str    
 1   site_id               122647 non-null  str    
 2   site_type             122647 non-null  str    
 3   site_name             122647 non-null  str    
 4   consumption_kwh       119807 non-null  float64
 5   consumption_euros     120160 non-null  float64
 6   temperature_celsius   119231 non-null  float64
 7   humidity_percent      119224 non-null  float64
 8   solar_irradiance_wm2  118683 non-null  float64
 9   hour                  122647 non-null  int64  
 10  day_of_week           122647 non-null  int64  
 11  day_name              122647 non-null  str    
 12  month                 122647 non-null  int64  
 13  is_weekend            122647 non-null  int64  
 14  is_working_hours      122647 non-null  int64  
dtypes: float64(

In [36]:
df.isna().sum()

timestamp                  0
site_id                    0
site_type                  0
site_name                  0
consumption_kwh         2840
consumption_euros       2487
temperature_celsius     3416
humidity_percent        3423
solar_irradiance_wm2    3964
hour                       0
day_of_week                0
day_name                   0
month                      0
is_weekend                 0
is_working_hours           0
dtype: int64

In [37]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["timestamp"].min(), df["timestamp"].max()

(Timestamp('2023-01-01 00:00:00'), Timestamp('2024-12-31 00:00:00'))

In [38]:
sites_summary = (
    df.groupby(["site_id", "site_name", "site_type"])
    .agg(
        date_debut=("timestamp", "min"),
        date_fin=("timestamp", "max"),
        nombre_lignes=("timestamp", "size")
    )
    .reset_index()
)

sites_summary

,site_id,site_name,site_type,date_debut,date_fin,nombre_lignes
0,SITE001,Bureau Tertiaire,office,2023-01-01,2024-12-31,17521
1,SITE002,Usine de Production,factory,2023-01-01,2024-12-31,17521
2,SITE003,Data Center,datacenter,2023-01-01,2024-12-31,17521
3,SITE004,Centre Commercial,retail,2023-01-01,2024-12-31,17521
4,SITE005,Hôpital,hospital,2023-01-01,2024-12-31,17521
5,SITE006,Bureau Tertiaire,office,2023-01-01,2024-12-31,17521
6,SITE007,Usine de Production,factory,2023-01-01,2024-12-31,17521


In [39]:
# Copie de travail pour conserver le dataset brut intact
df_clean = df.copy()

# Tri chronologique par site
df_clean = df_clean.sort_values(
    ["site_id", "timestamp"]
).reset_index(drop=True)

In [40]:
df_clean.duplicated(
    subset=["site_id", "timestamp"]
).sum()

np.int64(0)

In [41]:
missing_by_site = (
    df_clean
    .groupby("site_id")[
        [
            "consumption_kwh",
            "temperature_celsius",
            "humidity_percent",
            "solar_irradiance_wm2"
        ]
    ]
    .apply(lambda x: x.isna().sum())
)

missing_by_site

,consumption_kwh,temperature_celsius,humidity_percent,solar_irradiance_wm2
site_id,,,,
SITE001,411,491,494,566
SITE002,407,491,489,569
SITE003,407,488,491,571
SITE004,405,485,487,576
SITE005,409,493,493,559
SITE006,398,491,489,558
SITE007,403,477,480,565


In [60]:
columns = [
    "site_id",
    "timestamp",
    "site_type",
    "consumption_kwh",
    "temperature_celsius",
    "humidity_percent",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_working_hours"
]

df_clean = df[columns].copy()

In [61]:
df_clean.head()

,site_id,timestamp,site_type,consumption_kwh,temperature_celsius,humidity_percent,hour,day_of_week,month,is_weekend,is_working_hours
0,SITE001,2023-01-01 00:00:00,office,85.82,10.6,76.7,0,6,1,1,0
1,SITE001,2023-01-01 01:00:00,office,89.84,12.0,70.6,1,6,1,1,0
2,SITE001,2023-01-01 02:00:00,office,80.62,13.8,62.2,2,6,1,1,0
3,SITE001,2023-01-01 03:00:00,office,92.67,9.4,66.0,3,6,1,1,0
4,SITE001,2023-01-01 04:00:00,office,72.50,14.8,67.2,4,6,1,1,0


In [62]:
df.shape

(122647, 15)

In [63]:
# Conservation de la valeur brute
df_clean["consumption_kwh_corrected"] = df_clean["consumption_kwh"]
df_clean["temperature_celsius_corrected"] = df_clean["temperature_celsius"]
df_clean["humidity_percent_corrected"] = df_clean["humidity_percent"]

# Remplacement des valeurs manquantes avec la dernière valeur connue du même site
df_clean["consumption_kwh_corrected"] = (
    df_clean
    .groupby("site_id")["consumption_kwh_corrected"]
    .ffill()
)

df_clean["temperature_celsius_corrected"] = (
    df_clean
    .groupby("site_id")["temperature_celsius_corrected"]
    .ffill()
)

df_clean["humidity_percent_corrected"] = (
    df_clean
    .groupby("site_id")["humidity_percent_corrected"]
    .ffill()
)

# .ffill() = forward fill : quand une valeur est vide, Pandas reprend la dernière valeur connue juste avant.

In [64]:
df_clean[
    ["consumption_kwh_corrected","temperature_celsius_corrected","humidity_percent_corrected"]
].isna().sum()

consumption_kwh_corrected        0
temperature_celsius_corrected    0
humidity_percent_corrected       0
dtype: int64

In [66]:
df_clean.head()

,site_id,timestamp,site_type,consumption_kwh,temperature_celsius,humidity_percent,hour,day_of_week,month,is_weekend,is_working_hours,consumption_kwh_corrected,temperature_celsius_corrected,humidity_percent_corrected
0,SITE001,2023-01-01 00:00:00,office,85.82,10.6,76.7,0,6,1,1,0,85.82,10.6,76.7
1,SITE001,2023-01-01 01:00:00,office,89.84,12.0,70.6,1,6,1,1,0,89.84,12.0,70.6
2,SITE001,2023-01-01 02:00:00,office,80.62,13.8,62.2,2,6,1,1,0,80.62,13.8,62.2
3,SITE001,2023-01-01 03:00:00,office,92.67,9.4,66.0,3,6,1,1,0,92.67,9.4,66.0
4,SITE001,2023-01-01 04:00:00,office,72.50,14.8,67.2,4,6,1,1,0,72.50,14.8,67.2
